# Literature Scanning: Step 2 - Extracting data from papers with LLMs

In [1]:
import pandas as pd
from discovery_utils.utils.llm import batch_check

import pandas as pd
import os

# Alternative: s3://discovery-iss/data/afs_scanning/afs_open_alex_scan_parenting_interventions.csv
INPUT_DATA = 'afs_open_alex_scan_parenting_interventions.csv'

data_df = (
    pd.read_csv(INPUT_DATA)
    .query("publication_year >= 2000")
    .assign(text = lambda df: df['title'] + ' ' + df['abstract'])
)
len(data_df)

# testing on a smaller sample for now
# data_df = data_df.sample(100, random_state=42)

34145

In [2]:
from ast import literal_eval

def extract_author_countries(authorships):
    try:
        # Convert the string representation of the list to an actual list
        authors = literal_eval(authorships)
        countries = []
        for author in authors:
            for institution in author.get('institutions', []):
                if 'country_code' in institution:
                    countries.append(institution['country_code'])
        # remove null values
        countries = [country for country in countries if country is not None]
        return sorted(list(set(countries)))
    except (ValueError, SyntaxError):
        # Handle the case where the string cannot be evaluated
        return []

## Run the LLM pipeline

In [ ]:
_data_df = data_df
ids = _data_df.id.tolist()
text = data_df.text.tolist()
test_data = dict(zip(ids, text))

In [5]:
len(test_data)

34145

In [22]:
system_message = """
    Extract structured information about a parenting programme from the provided text.
    We are looking for programmes that are evidence-based (i.e. rigorously evaluated) and targeted at parents from pregnancy up to when their
    child is 5 years old.

    The programme could be relevant to developmental, behavioural, or well-being outcomes
    such as children language development, cognitive development, social-emotional skills, physical and mental health, or
    parental outcomes such as confidence, skills, behaviours, knowledge, self-efficacy, and well-being.
    
    Unless requested otherwise, adhere as precisely as possible to the language and text that is used in the provided text document.
    
    If the requested information is not described, return N/A. DO NOT make up any false information or false inferences.
"""

fields = [
    # Relevance
    {"name": "is_relevant", "type": "str", "description": "One-word answer: 'yes' if the text is about a parenting programme targeted at parents at any stage from pregnancy up to when the child is 5 years old, otherwise 'no'."},
    {"name": "is_relevant_reason", "type": "str", "description": "Short explanation (one sentence, 20 words) of why the text is relevant or not."},
    # Details about the programme
    {"name": "programme_name", "type": "str", "description": "The name of the programme or intervention."},
    {"name": "summary", "type": "str", "description": "A brief summary of what the programme involves and what it aims to achieve."},
    
    {"name": "child_outcomes_targeted", "type": "list[str]", "description": "Child-related outcomes the programme seeks to improve (e.g., 'language', 'cognitive development', 'social-emotional skills')."},
    {"name": "child_outcomes_category", "type": "list[str]", "description": "Infer if the child-related outcomes are best categorised in one of the following: 'Physical health and development', 'Mental health', 'Social, emotional and behavioral', 'Cognitive development', 'Speech, language and communication', 'N/A'."},
    {"name": "parent_outcomes_targeted", "type": "list[str]", "description": "Parent-related outcomes the programme targets (e.g., 'confidence', 'knowledge', 'wellbeing', 'skills')."},
    {"name": "parent_outcomes_category", "type": "list[str]", "description": "Infer if the parent-related outcomes are best categorised in one of the following: 'Parental behaviours and skills', 'Parental beliefs and knowledge', 'Parental well-being and mental health', 'N/A'."},    

    # Digital
    {"name": "digital_component", "type": "str", "description": "One-word answer: 'yes' if programme includes a digital component (e.g., text messages, online modules), otherwise 'no'."},
    {"name": "digital_component_description", "type": "str", "description": "Brief description of the digital component, if any."},
    
    # Context
    {"name": "country", "type": "str", "description": "The country or countries where the programme was delivered."},

    {"name": "age_range", "type": "str", "description": "The programme is aimed at parents of what age group? E.g., 'pregnancy','0-5 years', 'infants', 'toddlers'."},
    {"name": "age_range_numerical", "type": "str", "description": "Infer the probable age range in years and in numerical format (e.g., '0-5', '0-2', '3-5'). This should be '-1' if the programme is aimed at pregnancy."},

    {"name": "target_population", "type": "str", "description": "Description of the parent population the programme is aimed at (e.g., low-income mothers, ethnic minorities, parents under 25)."},    
    {"name": "disadvantaged_groups_engaged", "type": "list[str]", "description": "List of any specific disadvantaged groups the programme has successfully engaged, if described. Otherwise return N/A"},

    # Engagement details
    {"name": "is_about_engagement", "type": "str", "description": "One-word answer: 'yes' if the text is specifically about evaluating a strategy for engaging or recruiting families into interventions, otherwise 'no'."},

    {"name": "engagement_strategies", "type": "list[str]", "description": "Recruitment and engagement strategies used, if described."},
    {"name": "engagement_success", "type": "str", "description": "Any information about how successful the engagement strategies were."},
    {"name": "barriers_to_engagement", "type": "list[str]", "description": "Barriers or challenges mentioned regarding recruitment or engagement."},
    {"name": "other_engagement_learnings", "type": "list[str]", "description": "Other useful insights about recruitment, or engagement, if any."},
    
    # More details about the study
    {"name": "study_type", "type": "str", "description": "If possible, identify the type of study. Provide only the letter a-i: (a) Purely cross-sectional study without control variables, (b) Study measures outcome pre and post without control variables, (c) Purely cross-section study, uses control variables, (d) Study measures outcomes pre and post, uses control variables or matching tecniques, (e) Comparison of outcomes in treated group and comparison group, with counterfactuals (f) Quasi-experimental study, (g) Randomised controlled trial, (h) Meta-analysis, (i) N/A."},
    {"name": "programme_evaluation_desc", "type": "str", "description": "Description of how the programme was evaluated."},
    {"name": "evidence_of_effectiveness", "type": "list[str]", "description": "Evidence of outcomes, effectiveness, or evaluations cited."},
    {"name": "limitations", "type": "list[str]", "description": "Any limitations of the programme or study mentioned."},
]

In [ ]:
processor = batch_check.LLMProcessor(
    model_name="gpt-4.1-mini",
    # model_name="gpt-4o-mini",
    temperature=0,
    output_path="afs_open_alex_scan_2.jsonl",
    system_message=system_message,
    session_name="afs_open_alex_scan",
    output_fields=fields,
)

processor.run(test_data, batch_size=30, sleep_time=0.5)

2025-05-16 17:10:23,044 - root - INFO - Using OpenAI


<Task pending name='Task-5' coro=<LLMProcessor.process_text_data() running at /Users/karlis.kanders/Code/discovery_utils/discovery_utils/utils/llm/batch_check.py:120>>

2025-05-16 17:10:23,209 - root - INFO - Processing batch 1/1130
2025-05-16 17:10:33,929 - root - INFO - Processing batch 2/1130
2025-05-16 17:10:42,435 - root - INFO - Processing batch 3/1130
2025-05-16 17:10:53,297 - root - INFO - Processing batch 4/1130
2025-05-16 17:11:04,708 - root - INFO - Processing batch 5/1130
2025-05-16 17:11:19,128 - root - INFO - Processing batch 6/1130
2025-05-16 17:11:27,379 - root - INFO - Processing batch 7/1130
2025-05-16 17:11:37,641 - root - INFO - Processing batch 8/1130
2025-05-16 17:11:49,584 - root - INFO - Processing batch 9/1130
2025-05-16 17:12:02,225 - root - INFO - Processing batch 10/1130
2025-05-16 17:12:11,271 - root - INFO - Processing batch 11/1130
2025-05-16 17:12:18,857 - root - INFO - Processing batch 12/1130
2025-05-16 17:12:31,323 - root - INFO - Processing batch 13/1130
2025-05-16 17:12:43,336 - root - INFO - Processing batch 14/1130
2025-05-16 17:12:52,509 - root - INFO - Processing batch 15/1130
2025-05-16 17:13:01,154 - root - I

## Tidy up the extracted data before upload

In [5]:
output_df = pd.read_json("afs_open_alex_scan_2.jsonl", lines=True)
output_df

,is_relevant,is_relevant_reason,programme_name,summary,child_outcomes_targeted,child_outcomes_category,parent_outcomes_targeted,parent_outcomes_category,digital_component,digital_component_description,...,barriers_to_engagement,other_engagement_learnings,study_type,programme_evaluation_desc,evidence_of_effectiveness,limitations,id,timestamp,model,temperature
0,no,The programme targets adolescent girls pre-pre...,Ntshembo intervention,A community-based health promotion interventio...,[N/A],[N/A],[N/A],[N/A],no,N/A,...,[],[],i,N/A,[],[],https://openalex.org/W2113761365,2025-05-16 16:04:58.393144+00:00,gpt-4.1-mini,0
1,no,The programme targets parents of children aged...,Sexuality Education Intervention for Parents o...,A two-week sexuality education intervention ai...,[sexuality education knowledge],[N/A],"[attitude and beliefs, sexual communication, k...","[Parental beliefs and knowledge, Parental beha...",no,N/A,...,[],[],b,Within-subjects design with 15 parents receivi...,"[Significant improvements in attitude, beliefs...","[Small sample size (15 parents), No control gr...",https://openalex.org/W2951570962,2025-05-16 16:04:58.394702+00:00,gpt-4.1-mini,0
2,no,The text focuses on vaccination refusal and pe...,N/A,N/A,[],[N/A],[],[N/A],no,N/A,...,[],[],i,N/A,[],[],https://openalex.org/W3048469179,2025-05-16 16:04:58.395090+00:00,gpt-4.1-mini,0
3,no,The text describes a broad public health inter...,N/A,N/A,[],[],[],[],no,N/A,...,[],[],i,The Intervention Wheel model was subjected to ...,[The critique process established the validity...,[],https://openalex.org/W2043137199,2025-05-16 16:04:58.395247+00:00,gpt-4.1-mini,0
4,no,The text is a study examining associations amo...,N/A,N/A,[],[],[],[],no,N/A,...,[],[],c,Observational and self-report measures with 83...,[],[],https://openalex.org/W2134310657,2025-05-16 16:04:58.395356+00:00,gpt-4.1-mini,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34140,no,The text analyzes single fathers' child-care t...,N/A,N/A,[],[],[],[],no,N/A,...,[],[],c,Analysis of nationally representative American...,[],[],https://openalex.org/W2044699211,2025-05-16 20:16:56.569013+00:00,gpt-4.1-mini,0
34141,no,The text discusses the impact of suicide on fa...,N/A,N/A,[],[N/A],[],[N/A],no,N/A,...,[],[],i,N/A,[],[],https://openalex.org/W2098758147,2025-05-16 20:16:56.570124+00:00,gpt-4.1-mini,0
34142,no,The text is a systematic review about maternal...,N/A,N/A,[N/A],[N/A],[N/A],[N/A],no,N/A,...,[N/A],[N/A],h,Systematic literature review of 14 articles ex...,[Mixed findings: no association with children'...,[Findings are mixed; need for prospective desi...,https://openalex.org/W2592184171,2025-05-16 20:16:56.570407+00:00,gpt-4.1-mini,0
34143,no,The text focuses on parental caregivers of chi...,N/A,N/A,[],[],[],[],no,N/A,...,[],[],i,N/A,[],[],https://openalex.org/W2059170206,2025-05-16 20:16:56.570612+00:00,gpt-4.1-mini,0


In [11]:
final_df = (
    output_df
    .query("is_relevant == 'yes' or is_about_engagement == 'yes'")
)

In [12]:
output_df['is_relevant'].value_counts()

is_relevant
no     30306
yes     3839
Name: count, dtype: int64

In [13]:
output_df.columns

Index(['is_relevant', 'is_relevant_reason', 'programme_name', 'summary',
       'child_outcomes_targeted', 'child_outcomes_category',
       'parent_outcomes_targeted', 'parent_outcomes_category',
       'digital_component', 'digital_component_description', 'country',
       'age_range', 'age_range_numerical', 'target_population',
       'disadvantaged_groups_engaged', 'is_about_engagement',
       'engagement_strategies', 'engagement_success', 'barriers_to_engagement',
       'other_engagement_learnings', 'study_type', 'programme_evaluation_desc',
       'evidence_of_effectiveness', 'limitations', 'id', 'timestamp', 'model',
       'temperature'],
      dtype='object')

In [7]:
# output_df.groupby(['is_relevant', 'programme_evaluation']).size().reset_index(name='counts')

In [17]:
final_cols = [
       'programme_name',
       'doi',   
       'publication_year',  
       'cited_by_count',     
       'is_relevant',
       'is_relevant_reason',    
       # programme       
       'summary',           
       'child_outcomes_targeted',
       'child_outcomes_category',
       'parent_outcomes_targeted',
       'parent_outcomes_category',
       # digital
       'digital_component',
       'digital_component_description',
       # context
       'country',
       'author_countries',
       'age_range',
       'age_range_numerical',
       'target_population',
       'disadvantaged_groups_engaged',
       # engagement
       'is_about_engagement',
       'engagement_strategies',
       'engagement_success',
       'barriers_to_engagement',
       'other_engagement_learnings',
       # study type
       'study_type',
       'programme_evaluation_desc',
       'evidence_of_effectiveness',
       'limitations',
       # other metadata
       'id',
       'title', 
       'text',       
]

In [19]:
df_checked = (
    pd.read_json("afs_open_alex_scan_2.jsonl", lines=True)
    .query("is_relevant == 'yes' or is_about_engagement == 'yes'")
    .merge(data_df[['id', 'publication_year', 'authorships', 'doi', 'title', 'text', 'is_retracted', 'cited_by_count']], on='id', how='left')
    .assign(
        author_countries = lambda df: df['authorships'].apply(extract_author_countries),
    )
    .assign(
        child_outcomes = lambda df: df['child_outcomes_targeted'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        child_outcomes_category = lambda df: df['child_outcomes_category'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        parent_outcomes = lambda df: df['parent_outcomes_targeted'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        parent_outcomes_category = lambda df: df['parent_outcomes_category'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        disadvantaged_groups_engaged = lambda df: df['disadvantaged_groups_engaged'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        engagement_strategies = lambda df: df['engagement_strategies'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        barriers_to_engagement = lambda df: df['barriers_to_engagement'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        evidence_of_effectiveness = lambda df: df['evidence_of_effectiveness'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        limitations = lambda df: df['limitations'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
        other_learnings = lambda df: df['other_engagement_learnings'].apply(lambda x: ', '.join(x) if isinstance(x, list) else x),
    )
    .query("is_retracted == False")
)[final_cols]



In [24]:
len(df_checked)

5084

In [23]:
extra_fields = [
        {"name": "doi", "description": "The DOI of the publication."},
        {"name": "cited_by_count", "description": "The number of times the publication has been cited."},
        {"name": "text", "description": "The text of the publication."},
        {"name": "publication_year", "description": "The year of publication."},
        {"name": "id", "description": "The unique identifier of the publication."},
        {"name": "title", "description": "The title of the publication."},
    ]
all_fields = fields + extra_fields

# Create a table with explanations of all the fields used in the output table (cols columns)
df_explanations = pd.DataFrame.from_records(all_fields).drop(columns=['type'])
# Order it in the same order as the columns in the output table
df_explanations = df_explanations.set_index('name').reindex(final_cols).reset_index()
df_explanations

,name,description
0,programme_name,The name of the programme or intervention.
1,doi,The DOI of the publication.
2,publication_year,The year of publication.
3,cited_by_count,The number of times the publication has been c...
4,is_relevant,One-word answer: 'yes' if the text is about a ...
5,is_relevant_reason,"Short explanation (one sentence, 20 words) of ..."
6,summary,A brief summary of what the programme involves...
7,child_outcomes_targeted,Child-related outcomes the programme seeks to ...
8,child_outcomes_category,Infer if the child-related outcomes are best c...
9,parent_outcomes_targeted,Parent-related outcomes the programme targets ...


## Upload to Google Sheets (NB: don't overwrite)

In [ ]:
# commented out for now, so that it doesn't overwrite by accident as the data is already uploaded
# from discovery_utils.utils import (
#     google
# )
# sheet_id = "18IHEOjdyyGejZ5SFFcfKYfgCFTjNp9k_-bChT0PRlLU"

In [29]:
google.upload_data_to_gsheet(sheet_id, {"data": df_checked})
google.format_gsheet(sheet_id, "data", freeze_cols=2)

2025-05-19 15:59:22,661 - root - INFO - Connected to Google Sheet: Review of strategies to engage disadvantaged families in parenting interventions [19-05-2025]
2025-05-19 15:59:25,256 - root - INFO - Uploading DataFrame to sheet: data
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-05-19 16:01:12,149 - root - INFO - Upload completed successfully.
2025-05-19 16:01:22,069 - root - INFO - Connected to Google Sheet: Review of strategies to engage disadvantaged families in parenting interventions [19-05-2025]


In [30]:
google.upload_data_to_gsheet(sheet_id, {"columns": df_explanations})
google.format_gsheet(sheet_id, "columns", freeze_cols=0)

2025-05-19 16:01:33,087 - root - INFO - Connected to Google Sheet: Review of strategies to engage disadvantaged families in parenting interventions [19-05-2025]
2025-05-19 16:01:35,245 - root - INFO - Uploading DataFrame to sheet: columns
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.12/site-packages/df2gspread/df2gspread.py:138: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(str)
2025-05-19 16:01:48,376 - root - INFO - Upload completed successfully.
2025-05-19 16:01:50,158 - root - INFO - Connected to Google Sheet: Review of strategies to engage disadvantaged families in parenting interventions [19-05-2025]
